# Genomic Intelligence

Seven tools over the hosted Genomic Intelligence `/v1` API: promoter, splice,
enhancer, chromatin and annotation scoring, gene-expression prediction, and a
workflow that finds genes in a locus and scores expression for each one.

Inference runs on the vendor's service, so there is no model to download and no
GPU required. Set `GI_API_KEY` before running; request a key at
<https://genomicintelligence.ai>.

Research and development use. Not for clinical or diagnostic decisions.

**Outputs below were produced against Genomic Intelligence service version
`2026.08.19.5`.** Predictions move as models are retrained; re-run the notebook
rather than treating the printed numbers as current.

## Setup

Every tool takes `sequences` and a config. Sequence length bounds are per task
and are checked locally against the endpoint's published floor before any
request is sent.

In [1]:
import os

from proto_tools.tools.sequence_scoring.genomic_intelligence import (
    GIAnnotationConfig,
    GIAnnotationInput,
    GIExpressionConfig,
    GIExpressionInput,
    GIFindGenesConfig,
    GIFindGenesInput,
    GIPromoterConfig,
    GIPromoterInput,
    GISpliceConfig,
    GISpliceInput,
    run_gi_annotation,
    run_gi_expression,
    run_gi_find_genes_and_predict_expression,
    run_gi_promoter,
    run_gi_splice,
)

assert os.environ.get("GI_API_KEY"), "set GI_API_KEY before running this notebook"

with open("hbb_locus.txt") as handle:
    HBB_LOCUS = handle.read().strip()

print(f"HBB locus: {len(HBB_LOCUS):,} bp (GRCh38 chr11:5,220,000-5,245,000, gene-sense)")

HBB locus: 25,001 bp (GRCh38 chr11:5,220,000-5,245,000, gene-sense)


## Promoter

Slides the model across the sequence and reports windows called as promoters.
Coordinates are 0-based with exclusive ends.

In [2]:
promoter = run_gi_promoter(
    GIPromoterInput(sequences=HBB_LOCUS[:5000]),
    GIPromoterConfig(),
)
result = promoter.results[0]
print(f"model            {result.meta.model}")
print(f"windows scored   {result.total_windows}")
print(f"called promoter  {result.promoter_windows}")
print(f"max probability  {result.max_probability:.4f}")

model            g0-promoter-2000bp
windows scored   5
called promoter  1
max probability  0.9504


## Splice sites

The model is strand-specific. Submit transcript orientation: the opposite strand
returns sites at different positions, often at high confidence, so the result
cannot be checked for orientation after the fact.

In [3]:
splice = run_gi_splice(
    GISpliceInput(sequences=HBB_LOCUS[:5000]),
    GISpliceConfig(threshold=0.5),
)
result = splice.results[0]
print(f"sites  {result.total_sites}  ({result.donor_sites} donor, {result.acceptor_sites} acceptor)")
for site in result.sites[:5]:
    print(f"  {site.site_type:<9} {site.start:>6}-{site.end:<6} {site.score:.4f}")

sites  2  (1 donor, 1 acceptor)
  acceptor    1668-1674   0.9998
  donor       1889-1896   0.9999


## Annotation

Finds transcripts de novo, with no reference. Detection is strand-insensitive:
genes on either strand are found from one submission, and the reported `strand`
is relative to the sequence as submitted.

In [4]:
annotation = run_gi_annotation(
    GIAnnotationInput(sequences=HBB_LOCUS),
    GIAnnotationConfig(),
)
result = annotation.results[0]
print(f"transcripts  {result.total_transcripts}")
for transcript in result.transcripts[:5]:
    print(f"  {transcript.name:<14} {transcript.start:>6}-{transcript.end:<6} "
          f"strand {transcript.strand}  TSS {transcript.tss_position}")

transcripts  2
  transcript_1    10516-12165  strand +  TSS 10516
  transcript_2    17927-19534  strand +  TSS 17927


## Expression

The model scores exactly one 9,198 bp window centred on a TSS. Submit that
window, or a longer locus plus `tss_index` and let the service cut it — the
window actually scored is echoed back.

`description` is conditioning text fed to the model, not a label. Its wording
changes the prediction, so hold it fixed across runs you intend to compare.

In [5]:
tss = result.transcripts[0].tss_position if result.transcripts else len(HBB_LOCUS) // 2

expression = run_gi_expression(
    GIExpressionInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB", "tss_index": tss}]),
    GIExpressionConfig(),
)
prediction = expression.results[0]
print(f"tss_index applied  {prediction.tss_index}")
print(f"window scored      {prediction.scored_window}")
print(f"log(TPM+1)         {prediction.expression_log_tpm:.4f}")
print(f"TPM                {prediction.expression_tpm:.4f}")

tss_index applied  10516
window scored      [5917, 15115]
log(TPM+1)         1.0703
TPM                1.9163


### Conditioning text changes the prediction

The same sequence under two descriptions. This is why the wording has to be held
fixed when comparing runs.

In [6]:
for description in (
    "assay term name is polyA plus RNA-seq. biosample summary is Homo sapiens K562.",
    "K562",
):
    out = run_gi_expression(
        GIExpressionInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB", "tss_index": tss}]),
        GIExpressionConfig(description=description),
    )
    print(f"{out.results[0].expression_log_tpm:8.4f}  <- {description[:60]}")

  1.0703  <- assay term name is polyA plus RNA-seq. biosample summary is 


  0.1426  <- K562


## Find genes and predict expression

Annotation and expression in one call, centring each window on the gene's own
TSS. Use it when the TSS positions are not known in advance.

This endpoint refuses synchronous delivery above 50,000 bp; the tool switches to
the polling path automatically. Its length ceiling is the endpoint's own
500,000 bp, which is not the expression model's window.

In [7]:
workflow = run_gi_find_genes_and_predict_expression(
    GIFindGenesInput(sequences=[{"sequence": HBB_LOCUS, "name": "HBB"}]),
    GIFindGenesConfig(),
)
result = workflow.results[0]
print(f"genes found   {result.genes_found}")
print(f"genes scored  {result.genes_scored}")
for gene in result.predictions:
    if gene.skipped:
        print(f"  {gene.gene_name:<14} skipped: {gene.skip_reason}")
    else:
        print(f"  {gene.gene_name:<14} TSS {gene.tss_position:>6}  log(TPM+1) {gene.expression:.4f}")

genes found   2
genes scored  2
  transcript_1   TSS  10516  log(TPM+1) 1.0703
  transcript_2   TSS  17927  log(TPM+1) 0.9570


## Scoring many sequences

Every tool takes a list and returns one result per input, in order, which is the
shape a Constraint or Optimizer consumes when scoring a population.

In [8]:
variants = [
    {"sequence": HBB_LOCUS[offset : offset + 3000], "name": f"window_{offset}"}
    for offset in (0, 6000, 12000)
]
batch = run_gi_promoter(GIPromoterInput(sequences=variants), GIPromoterConfig())
for item in batch.results:
    print(f"{item.name:<14} max probability {item.max_probability:.4f}")

window_0       max probability 0.9504
window_6000    max probability 0.0117
window_12000   max probability 0.2551
